# kaggle-vllm 0.1.1 — Clean Dual-T4 Bootstrap + Inference Acceptance Test

This notebook is designed to avoid the previous error:

`refusing to overwrite non-empty staged wheel destination: /kaggle/working/vllm-staged`

Instead of reusing the default staging directories, it creates a **notebook-owned runtime root** and removes only that root before bootstrapping.

Validated delivery path:

**PyPI `kaggle-vllm==0.1.1` → Hugging Face pinned native wheel → checksum verification → staged runtime → 2× Tesla T4 → TP=2 inference**

Canonical resources:

- PyPI SDK: `https://pypi.org/project/kaggle-vllm/`
- Native binaries: `https://huggingface.co/waqasm86/kaggle-vllm-binaries`
- TP=2 Qwen sharded state: `https://huggingface.co/waqasm86/kaggle-vllm-models`

### Kaggle settings

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Add a Kaggle secret named **HF_TOKEN** if you want authenticated Hugging Face downloads.
  The repositories are public, so the token is helpful but not required.

In [1]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kaggle:", os.path.exists("/kaggle"))
print("\nGPUs:")
subprocess.run(["nvidia-smi", "-L"], check=True)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Kaggle: True

GPUs:
GPU 0: Tesla T4 (UUID: GPU-e1585696-8ff1-0eb8-fc5f-996cbb53c30f)
GPU 1: Tesla T4 (UUID: GPU-6e8a5b24-e121-d60d-bebe-b2e4588d4ed6)


CompletedProcess(args=['nvidia-smi', '-L'], returncode=0)

## 1. Load `HF_TOKEN` securely from Kaggle Secrets

The token is placed into environment variables used by Hugging Face clients and inherited by subprocesses.

**The token value is never printed.**

In [2]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN loaded from Kaggle Secrets: YES")
    else:
        print("HF_TOKEN loaded: NO (public repositories can still be used)")
except Exception as exc:
    print("HF_TOKEN unavailable; continuing with public access.")
    print("Reason type:", type(exc).__name__)

HF_TOKEN loaded from Kaggle Secrets: YES


## 2. Install the lightweight SDK from PyPI

Installing `kaggle-vllm` does **not** install vLLM, Torch, CUDA, or NVIDIA packages as normal dependencies.

In [3]:
%pip install --no-cache-dir --upgrade "kaggle-vllm[hub]==0.1.1"

Note: you may need to restart the kernel to use updated packages.


In [4]:
import kaggle_vllm

print("kaggle_vllm version:", kaggle_vllm.__version__)
assert kaggle_vllm.__version__ == "0.1.1"

subprocess.run(["kaggle-vllm", "fingerprint"], check=True)
subprocess.run(
    ["kaggle-vllm", "verify-gpus", "--tensor-parallel-size", "2"],
    check=True,
)

kaggle_vllm version: 0.1.1
{
  "is_kaggle": true,
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    }
  ],
  "nccl": "2.27.5",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "nvcc_version": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "cuda_home": "/usr/local/cuda",
  "cuda_driver": "/usr/local/nvidia/lib64/libcuda.so",
  "cmake_library_path": null
}
PASS: 

CompletedProcess(args=['kaggle-vllm', 'verify-gpus', '--tensor-parallel-size', '2'], returncode=0)

## 3. Create clean notebook-owned bootstrap destinations

The previous notebook failed because the default staging directory already contained files.

This cell deliberately uses:

`/kaggle/working/kaggle-vllm-e2e-011/`

and deletes **only that notebook-owned runtime root** before the new bootstrap.

The wheel download cache is kept separately so a valid previously downloaded native wheel can be reused after SHA256 verification.

In [5]:
RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-e2e-011")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"

# Reuse a checksum-verified cache across notebook retries.
CACHE = Path("/kaggle/working/kaggle-vllm-cache")

if RUNTIME_ROOT.exists():
    print("Removing previous notebook-owned runtime root:", RUNTIME_ROOT)
    shutil.rmtree(RUNTIME_ROOT)

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

print("STAGED :", STAGED)
print("OVERLAY:", OVERLAY)
print("CACHE  :", CACHE)
print("MANIFEST:", MANIFEST)

STAGED : /kaggle/working/kaggle-vllm-e2e-011/vllm-staged
OVERLAY: /kaggle/working/kaggle-vllm-e2e-011/vllm-runtime-overlay
CACHE  : /kaggle/working/kaggle-vllm-cache
MANIFEST: /kaggle/working/kaggle-vllm-e2e-011/kaggle-vllm-runtime.json


## 4. Strict dry-run against the clean custom paths

Expected identity:

- Repository: `waqasm86/kaggle-vllm-binaries`
- Revision: `f6b4f10de54924ed6fe9e28cceab84eca7276ab6`
- Native wheel SHA256: `5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c`

In [6]:
BOOTSTRAP_BASE = [
    "kaggle-vllm",
    "bootstrap",
    "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

dry = subprocess.run(BOOTSTRAP_BASE + ["--dry-run"], check=True)
print("dry-run return code:", dry.returncode)

profile: kaggle-t4x2-cu128
compatible: True
strict: True
HF repository: waqasm86/kaggle-vllm-binaries
immutable revision: f6b4f10de54924ed6fe9e28cceab84eca7276ab6
wheel: vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
expected SHA256: 5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c
cache: /kaggle/working/kaggle-vllm-cache
staged: /kaggle/working/kaggle-vllm-e2e-011/vllm-staged
overlay: /kaggle/working/kaggle-vllm-e2e-011/vllm-runtime-overlay
manifest: /kaggle/working/kaggle-vllm-e2e-011/kaggle-vllm-runtime.json
PASS: Python implementation: CPython
PASS: Python ABI: cp312
PASS: operating system: Linux
PASS: machine: x86_64
PASS: Kaggle runtime: True
PASS: PyTorch: 2.10.0+cu128
PASS: PyTorch CUDA: 12.8
PASS: visible GPU count: 2
PASS: GPU model: Tesla T4, Tesla T4
PASS: GPU compute capability: SM75, SM75
PASS: NCCL: 2.27.5
dry-run: no download or filesystem changes performed
would run: /usr/bin/python3 -m pip install --target /kaggle/working/kagg

## 5. Download, verify and stage the native CUDA vLLM runtime

This is the actual native-runtime installation step.

Because the staging and overlay destinations are clean, it will not hit the previous non-empty-destination refusal.

In [7]:
result = subprocess.run(BOOTSTRAP_BASE, check=True)
print("bootstrap return code:", result.returncode)

assert MANIFEST.is_file(), f"manifest was not created: {MANIFEST}"
assert STAGED.is_dir() and any(STAGED.iterdir()), "staged native runtime is empty"
assert OVERLAY.is_dir() and any(OVERLAY.iterdir()), "dependency overlay is empty"

print("Bootstrap manifest:", MANIFEST)
print("Native runtime staged successfully.")

Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 185.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 332.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 230.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 268.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 325.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 234.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 272.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 6. Activate the custom runtime inside this notebook process

Because this notebook intentionally used a custom manifest path, set `KAGGLE_VLLM_MANIFEST` and activate that exact runtime explicitly.

In [8]:
os.environ["KAGGLE_VLLM_MANIFEST"] = str(MANIFEST)

from kaggle_vllm.bootstrap import activate_runtime

activated = activate_runtime(MANIFEST)
print("Runtime activated:", activated)
assert activated

Runtime activated: True


## 7. Verify native vLLM imports

In [9]:
import importlib
import vllm

print("vLLM version:", getattr(vllm, "__version__", "unknown"))
print("vLLM path:", vllm.__file__)

for module_name in ("vllm._C", "vllm._moe_C", "vllm.cumem_allocator"):
    module = importlib.import_module(module_name)
    print("PASS:", module_name, "->", getattr(module, "__file__", "<built-in>"))

vLLM version: 0.18.2.dev0+ga26e8dc7f.d20260822
vLLM path: /kaggle/working/kaggle-vllm-e2e-011/vllm-staged/vllm/__init__.py
PASS: vllm._C -> /kaggle/working/kaggle-vllm-e2e-011/vllm-staged/vllm/_C.abi3.so
PASS: vllm._moe_C -> /kaggle/working/kaggle-vllm-e2e-011/vllm-staged/vllm/_moe_C.abi3.so
PASS: vllm.cumem_allocator -> /kaggle/working/kaggle-vllm-e2e-011/vllm-staged/vllm/cumem_allocator.abi3.so


## 8. Verify Kaggle's original Torch/CUDA stack remains intact

In [10]:
import torch

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

assert torch.__version__ == "2.10.0+cu128"
assert torch.version.cuda == "12.8"
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2

for i in range(torch.cuda.device_count()):
    print(
        i,
        torch.cuda.get_device_name(i),
        "SM", torch.cuda.get_device_capability(i),
        "VRAM GiB", round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2),
    )

Torch: 2.10.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU count: 2
0 Tesla T4 SM (7, 5) VRAM GiB 14.56
1 Tesla T4 SM (7, 5) VRAM GiB 14.56


## 9. Fast real TP=2 inference test with OPT-125M

This proves that the newly downloaded/staged native runtime can execute tensor-parallel inference across both T4 GPUs before attempting the much larger Qwen checkpoint.

In [11]:
from kaggle_vllm import KaggleLLM
from vllm import SamplingParams

llm = KaggleLLM(
    model="facebook/opt-125m",
    tensor_parallel_size=2,
    max_model_len=512,
    gpu_memory_utilization=0.60,
)

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=48,
)

outputs = llm.generate(
    ["Kaggle dual NVIDIA T4 tensor-parallel inference is"],
    sampling,
)

for output in outputs:
    print("PROMPT:", output.prompt)
    for candidate in output.outputs:
        print("OUTPUT:", candidate.text)

INFO 08-25 00:43:04 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': 'facebook/opt-125m'}


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 08-25 00:43:28 [model.py:533] Resolved architecture: OPTForCausalLM
INFO 08-25 00:43:28 [model.py:1582] Using max model len 512
INFO 08-25 00:43:29 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-25 00:43:29 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 08-25 00:43:29 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-25 00:43:29 [vllm.py:820] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-25 00:43:29 [vllm.py:985] Cudagraph is disabled under eager mode
INFO 08-25 00:43:29 [compilation.py:289] Enabled custom fusions: norm_quant, act_quant


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.54it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.54it/s]
(Worker_TP0 pid=268) 


(Worker_TP0 pid=268) INFO 08-25 00:44:17 [default_loader.py:384] Loading weights took 0.29 seconds
(Worker_TP0 pid=268) INFO 08-25 00:44:18 [gpu_model_runner.py:4566] Model loading took 0.12 GiB memory and 4.875034 seconds
(Worker_TP0 pid=268) INFO 08-25 00:44:40 [gpu_worker.py:456] Available KV cache memory: 8.39 GiB
(EngineCore pid=244) INFO 08-25 00:44:40 [kv_cache_utils.py:1316] GPU KV cache size: 488,736 tokens
(EngineCore pid=244) INFO 08-25 00:44:40 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 954.56x
(EngineCore pid=244) INFO 08-25 00:44:42 [core.py:281] init engine (profile, create kv cache, warmup model) took 23.83 seconds
(EngineCore pid=244) INFO 08-25 00:44:44 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=244) WARNING 08-25 00:44:44 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=244) WARNING 08-25 00:44:44 [vllm.py:82

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

PROMPT: Kaggle dual NVIDIA T4 tensor-parallel inference is
OUTPUT:  a powerful and fast inference engine that can be used to optimize the performance of your system. It is a powerful and fast inference engine that can be used to optimize the performance of your system. It is a powerful and fast inference engine that can


## 10. Download the canonical Qwen TP=2 sharded-state repository

This is the full project acceptance path and downloads several GB.

The Hugging Face repository contains the previously created vLLM-native TP=2 `sharded_state`.
It is not a normal Transformers weight layout.

Set `RUN_QWEN = False` if you only want the smaller native-runtime/TP=2 smoke test.

In [12]:
RUN_QWEN = True

QWEN_REPO = "waqasm86/kaggle-vllm-models"
QWEN_LOCAL = Path("/kaggle/working/kaggle-vllm-qwen-tp2")

if RUN_QWEN:
    from huggingface_hub import snapshot_download

    local_path = snapshot_download(
        repo_id=QWEN_REPO,
        local_dir=str(QWEN_LOCAL),
        token=HF_TOKEN,
    )
    print("Qwen snapshot:", local_path)

    subprocess.run(
        ["kaggle-vllm", "inspect-shards", str(QWEN_LOCAL)],
        check=True,
    )
else:
    print("Skipping Qwen download/load.")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

NOTICE:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

SHARDED_STATE_SHA256SUMS.txt:   0%|          | 0.00/392 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Qwen snapshot: /kaggle/working/kaggle-vllm-qwen-tp2
path: /kaggle/working/kaggle-vllm-qwen-tp2
TP ranks: 2
shards: 4
weight bytes: 6172262512
valid: True


## 11. Load the Qwen persistent sharded state at TP=2 and run inference

In [13]:
if RUN_QWEN:
    # Free the small smoke-test engine before allocating the larger Qwen engine.
    try:
        del llm
    except NameError:
        pass

    import gc
    gc.collect()
    torch.cuda.empty_cache()

    from kaggle_vllm import KaggleLLM
    from vllm import SamplingParams

    qwen = KaggleLLM(
        model=str(QWEN_LOCAL),
        tensor_parallel_size=2,
        load_format="sharded_state",
        max_model_len=2048,
        gpu_memory_utilization=0.70,
    )

    sampling = SamplingParams(
        temperature=0.2,
        top_p=0.9,
        max_tokens=96,
    )

    prompt = (
        "Explain in three concise sentences why tensor parallelism "
        "is useful for LLM inference."
    )

    outputs = qwen.generate([prompt], sampling)

    for output in outputs:
        print("PROMPT:", output.prompt)
        for candidate in output.outputs:
            print("OUTPUT:", candidate.text)
else:
    print("Qwen inference skipped.")

(EngineCore pid=244) INFO 08-25 00:45:45 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=244) INFO 08-25 00:45:45 [core.py:1224] Shutdown complete
(Worker_TP1 pid=269) INFO 08-25 00:45:45 [multiproc_executor.py:759] Parent process exited, terminating worker queues
(Worker_TP0 pid=268) INFO 08-25 00:45:45 [multiproc_executor.py:759] Parent process exited, terminating worker queues
(Worker_TP1 pid=269) INFO 08-25 00:45:45 [multiproc_executor.py:854] WorkerProc shutting down.
(Worker_TP0 pid=268) INFO 08-25 00:45:45 [multiproc_executor.py:854] WorkerProc shutting down.
INFO 08-25 00:45:50 [utils.py:233] non-default args: {'load_format': 'sharded_state', 'dtype': 'float16', 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': '/kaggle/working/kaggle-vllm-qwen-tp2'}
INFO 08-25 00:46:10 [model.py:533] Resolved architecture: Qwen2ForCausalLM
WARNING 08-25 00:

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

PROMPT: Explain in three concise sentences why tensor parallelism is useful for LLM inference.
OUTPUT:  Tensor parallelism distributes the model's parameters across multiple GPUs, enabling efficient parallel computation and reducing communication overhead. This approach accelerates inference by leveraging the parallel processing capabilities of modern hardware, thereby improving throughput and reducing latency. Additionally, tensor parallelism allows for scaling the model size without a proportional increase in computational resources, making it feasible to train and deploy larger, more powerful LLMs. Tensor parallelism is thus a key technique for enhancing the performance and scalability of large language models during


## 12. Final acceptance report

If all required cells above pass, you have freshly validated:

- PyPI `kaggle-vllm==0.1.1`
- Kaggle Python 3.12 / Torch 2.10.0+cu128
- 2× Tesla T4 / SM75
- canonical Hugging Face binary repository
- immutable native wheel revision
- SHA256-verified native runtime
- staged native extension imports
- TP=2 inference
- optional full Qwen TP=2 `sharded_state` reload and inference

In [14]:
print("=== kaggle-vllm final fingerprint ===")
subprocess.run(["kaggle-vllm", "fingerprint"], check=False)

print("\n=== runtime manifest ===")
print(MANIFEST)
print("exists:", MANIFEST.exists())

print("\n=== GPU state ===")
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.used,utilization.gpu",
        "--format=csv,noheader",
    ],
    check=False,
)

=== kaggle-vllm final fingerprint ===
{
  "is_kaggle": true,
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    }
  ],
  "nccl": "2.27.5",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "nvcc_version": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "cuda_home": "/usr/local/cuda",
  "cuda_driver": "/usr/local/nvidia/lib64/libcuda.so",
  "cmake_library_path": nu

CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.used,utilization.gpu', '--format=csv,noheader'], returncode=0)